# Automated Resume Screening & Candidate Ranking System
**Future Interns — Machine Learning Task 3**  
**Author:** Lohith Gopi  

### Objectives
1. **Resume Text Cleaning & Normalization:** Strip noise, links, symbols, preserve technical tokens.
2. **Skill Extraction & Matching:** Identify candidate skills and compare against job descriptions.
3. **Candidate Ranking:** Hybrid scoring combining TF-IDF semantic relevance and skill match percentage.
4. **Skill Gap Identification:** Highlight missing required competencies for automated recruiter feedback.

In [ ]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Load resume dataset
df = pd.read_csv('data/resumes.csv')
print(f'Total resumes: {len(df)}')
print(f'Categories ({df["Category"].nunique()}):', df['Category'].unique()[:8])
df.head(3)

## 1. Resume Dataset Distribution (EDA)

In [ ]:
plt.figure(figsize=(12, 5))
sns.countplot(data=df, y='Category', order=df['Category'].value_counts().index, palette='crest', hue='Category', legend=False)
plt.title('Distribution of Resumes by Industry Category')
plt.xlabel('Candidate Count')
plt.tight_layout()
plt.show()

## 2. Text Preprocessing & Skill Extraction Engine

In [ ]:
COMMON_SKILLS = {
    'python', 'r', 'sql', 'java', 'c++', 'c#', 'javascript', 'typescript', 'html', 'css',
    'react', 'node.js', 'django', 'flask', 'fastapi', 'spring', 'docker', 'kubernetes',
    'aws', 'azure', 'gcp', 'git', 'linux', 'machine learning', 'deep learning', 'nlp',
    'tensorflow', 'pytorch', 'scikit-learn', 'pandas', 'numpy', 'spark', 'hadoop',
    'tableau', 'power bi', 'agile', 'scrum', 'jira', 'ci/cd'
}

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'\S+@\S+', ' ', text)
    text = re.sub(r'[,;!?()[\]{}:"\']', ' ', text)
    text = re.sub(r'\.(?=\s|$)', ' ', text)
    text = re.sub(r'[^a-z0-9#+\-\./\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

def extract_skills(text: str, skill_set=COMMON_SKILLS) -> set:
    cleaned = f' {clean_text(text)} '
    detected = set()
    for skill in skill_set:
        pattern = r'(?<=\s)' + re.escape(skill) + r'(?=\s)'
        if re.search(pattern, cleaned):
            detected.add(skill)
    return detected

sample_resume = df['Resume'].iloc[0]
print('Sample Detected Skills:', sorted(list(extract_skills(sample_resume))))

## 3. Screening & Ranking Algorithm

In [ ]:
def rank_candidates(candidates_df: pd.DataFrame, job_description: str, required_skills: list = None) -> pd.DataFrame:
    clean_jd = clean_text(job_description)
    target_skills = set(s.lower() for s in required_skills) if required_skills else extract_skills(clean_jd)
    
    cleaned_resumes = [clean_text(r) for r in candidates_df['Resume']]
    corpus = [clean_jd] + cleaned_resumes
    
    tfidf = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))
    matrix = tfidf.fit_transform(corpus)
    cosine_scores = cosine_similarity(matrix[0:1], matrix[1:]).flatten()
    
    records = []
    for i, (_, row) in enumerate(candidates_df.iterrows()):
        cand_skills = extract_skills(row['Resume'])
        matched = target_skills.intersection(cand_skills)
        missing = target_skills.difference(cand_skills)
        skill_ratio = len(matched) / len(target_skills) if target_skills else 0.0
        composite = round(float((0.5 * cosine_scores[i] + 0.5 * skill_ratio) * 100), 2)
        
        records.append({
            'candidate_id': f'CAND_{i+1:03d}',
            'category': row['Category'],
            'composite_score': composite,
            'similarity_pct': round(float(cosine_scores[i] * 100), 2),
            'skill_match_pct': round(float(skill_ratio * 100), 2),
            'matched_skills': sorted(list(matched)),
            'missing_skills': sorted(list(missing))
        })
    
    ranked_df = pd.DataFrame(records).sort_values(by='composite_score', ascending=False).reset_index(drop=True)
    ranked_df.index += 1
    ranked_df.index.name = 'rank'
    return ranked_df

## 4. Live Screening Experiment: Data Scientist Role

In [ ]:
ds_job_desc = '''
We are seeking a Senior Data Scientist to design machine learning and deep learning pipelines.
Must be proficient in Python, SQL, Machine Learning, Pandas, Scikit-learn, and PyTorch.
Experience with AWS cloud deployment is a strong plus.
'''
ds_required = ['python', 'sql', 'machine learning', 'deep learning', 'pandas', 'scikit-learn', 'pytorch', 'aws']

ds_ranked = rank_candidates(df, ds_job_desc, ds_required)
print('Top 5 Ranked Candidates for Data Scientist Role:')
ds_ranked[['candidate_id', 'category', 'composite_score', 'skill_match_pct', 'matched_skills', 'missing_skills']].head(5)

## 5. Candidate Ranking & Skill Gap Visualizations

In [ ]:
top_10 = ds_ranked.head(10).copy()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(top_10['candidate_id'] + ' (' + top_10['category'] + ')', top_10['composite_score'], color='#2ecc71')
ax.invert_yaxis()
ax.set_xlabel('Composite Fit Score (0 - 100)')
ax.set_title('Top 10 Ranked Candidates for Data Scientist Opening')

for bar in bars:
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2, f'{bar.get_width():.1f}%', va='center', fontsize=9)

plt.xlim(0, 110)
plt.tight_layout()
plt.show()

## 6. Recruiter Feedback Report (Skill Gap Analysis)

In [ ]:
def print_recruiter_card(candidate_row):
    print(f"=== Candidate: {candidate_row['candidate_id']} ({candidate_row['category']}) ===")
    print(f"Overall Fit Score: {candidate_row['composite_score']:.1f}%")
    print(f"Skills Matched ({len(candidate_row['matched_skills'])}): {', '.join(candidate_row['matched_skills']) or 'None'}")
    print(f"Skill Gaps ({len(candidate_row['missing_skills'])}): {', '.join(candidate_row['missing_skills']) or 'None (100% Fit)'}")
    print("-" * 50)

for _, row in ds_ranked.head(3).iterrows():
    print_recruiter_card(row)